# 1. 파일 만들기

In [26]:
import os
import csv

file_path = "./hand_data.csv"

if not os.path.exists(file_path):
    with open(file_path, "w") as file:
        writer = csv.writer(file)

        writer.writerow([1,2,3,4,5])
        # csv 파일 확인해보기


In [27]:
# 파일에 데이터 추가하기
with open(file_path, "a") as file:
    writer = csv.writer(file)
    writer.writerow([5,4,3,2,1])

In [28]:
import sys
import cv2
import mediapipe as mp
import os
import csv

# hand landmark 옵션
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
# (New) 저장할 파일명 넣기 file_path
file_path = "hand_data.csv"
# (New) 만약에 파일 경로가 없으면 새로 만들기
if not os.path.exists(file_path):
    with open(file_path, "w") as file:
        writer = csv.writer(file)
# 웹캠
vcap = cv2.VideoCapture(0)
#   # 카메라 감지
while True:
    ret, frame = vcap.read()
    if not ret:
        print("카메라가 작동하지 않습니다.")
        sys.exit()

    # 좌우반전
    frame = cv2.flip(frame, 1)

#   # 손 그리기 준비
    frame.flags.writeable = True
#   # 손 감지
    results = hands.process(frame)
#   # Hand Landmark 추출
    if results.multi_hand_landmarks:
#       # 손 하나의 Hand Landmark 추출 및 그리기
        for hand_landmarks in results.multi_hand_landmarks:
            one_hand = results.multi_hand_landmarks[0]
#           # 저장 데이터 만들기, 포인트 그리기
            # 위치 모으기
            height, width, _ = frame.shape 

            landmark_list = []

            for landmark in one_hand.landmark:
                # 좌표 모으기
                landmark_list.extend([landmark.x, landmark.y, landmark.z])
                # 그리기
                point_x = int(landmark.x * width) # landmark.x 는 이미지에서의 비율적 위치(소수점)을 리턴하기 때문에 화면 넓이를 곱해줘야한다.
                point_y = int(landmark.y * height)

                cv2.circle(frame, (point_x, point_y), 5, (0, 255, 0), 2)
        # (New) 1누르면 rock, 2 누르면 scissors, 3누르면 paper 그리고 데이터 저장
        key = cv2.waitKey(1) #ASCII 코드
        if key == ord('1'):
            #정답라벨 추가
            landmark_list.append("rock")
            with open(file_path, "a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(landmark_list)
                cv2.putText(frame, "Save Rock Data!", (10, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 0, 0), 2)
        elif key == ord("2"):
            # 정답라벨 추가
            landmark_list.append("Sissors")
            #데이터 추가
            with open(file_path, "a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(landmark_list)
                cv2.putText(frame, "Save Sissors Data", (10, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 0, 0), 2)
        elif key == ord("3"):
            #정답라벨 추가
            landmark_list.append("Paper")
            #데이터추가
            with open(file_path, "a", newline="") as file:
                writer = csv.writer(file)
                writer.writerow(landmark_list)
                cv2.putText(frame, "Save Paper Data!", (10, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 0, 0), 2)
     ########################            
    # 화면 띄우기
    cv2.imshow("webcam", frame)

#   # ESC 누르면 종료
    key = cv2.waitKey(1)
    
    if key == 27: #ESC
        break


# 마무리

vcap.release()
cv2.destroyAllWindows()

# 1) 데이터 불러오기

In [29]:
import pandas as pd

file_path = "hand_data.csv"
data = pd.read_csv(file_path, header=None) 
data.head()
data

,0,1,2,3,4,5,6,7,8,9,...,54,55,56,57,58,59,60,61,62,63
0,0.305067,0.649124,-1.607328e-07,0.386241,0.589204,-0.006307,0.422951,0.533554,-0.022819,0.448303,...,0.408656,0.541817,-0.076810,0.400699,0.574792,-0.073208,0.378853,0.571938,-0.070210,rock
1,0.308457,0.630637,-1.063354e-07,0.320045,0.547446,0.006386,0.352151,0.478464,-0.006284,0.394727,...,0.443189,0.556675,-0.080030,0.430447,0.579231,-0.072914,0.410875,0.574730,-0.069774,rock
2,0.316663,0.632348,-3.342882e-07,0.303546,0.546254,0.003916,0.314688,0.475112,0.001074,0.343990,...,0.443863,0.534018,-0.018764,0.430773,0.545232,-0.006842,0.419035,0.561458,0.000907,rock
3,0.320264,0.596861,-3.014286e-07,0.338590,0.513418,0.014172,0.372984,0.474702,0.013766,0.408269,...,0.459747,0.587489,-0.012084,0.444166,0.594206,0.001143,0.426757,0.598463,0.008449,rock
4,0.400295,0.672929,-2.420993e-07,0.433110,0.602566,0.010781,0.475700,0.580001,0.009093,0.510188,...,0.521470,0.751851,-0.026549,0.500657,0.751275,-0.021382,0.484296,0.739897,-0.018277,rock
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,0.488150,0.443399,-4.821082e-07,0.420712,0.421579,-0.022073,0.364055,0.371970,-0.036996,0.340885,...,0.448117,0.327989,-0.053792,0.457747,0.364914,-0.048986,0.474772,0.355185,-0.040415,rock
494,0.539024,0.444619,-5.907611e-07,0.476347,0.417434,-0.021020,0.420029,0.350282,-0.032305,0.410189,...,0.532370,0.317724,-0.055170,0.533481,0.358762,-0.049941,0.549308,0.350456,-0.042110,rock
495,0.805678,0.703539,-3.399471e-07,0.700730,0.583702,-0.028038,0.607394,0.500687,-0.076082,0.527116,...,0.645913,0.761030,-0.196004,0.665738,0.772707,-0.189359,0.711245,0.738695,-0.184219,rock
496,0.774081,0.654239,-6.250143e-07,0.706881,0.516990,-0.012643,0.620096,0.404809,-0.049042,0.518592,...,0.560789,0.638600,-0.177811,0.585730,0.667805,-0.171543,0.626693,0.648157,-0.166200,rock


In [30]:
data[63].value_counts()

63
Paper      177
rock       162
Sissors    158
Name: count, dtype: int64

# 2) 데이터 분할하기

In [31]:
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# data[63] = le.fit_transform(data[63])
# print(le.classes_)

In [32]:
data[63] = data[63].apply(lambda x: 0 if x == "rock" else 1 if x == "Sissors" else 2)

In [33]:
data[63].value_counts()

63
2    178
0    162
1    158
Name: count, dtype: int64

In [34]:
from sklearn.model_selection import train_test_split

x = data.iloc[:, 0:63]
y = data.iloc[:, 63]
x_train, x_test , y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 3) 모델 만들기 및 평가

In [35]:
from xgboost import XGBClassifier
import numpy as np

model = XGBClassifier(n_estimators=100, max_depth=3)
model.fit(x_train, y_train)

score = model.score(x_test, y_test)

print("Accuracy:", score)
print("Accuracy Rate:", np.round(score*100, 2))

Accuracy: 0.98
Accuracy Rate: 98.0


In [36]:
from sklearn.metrics import classification_report

y_pred = model.predict(x_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        37
           1       0.96      0.96      0.96        27
           2       0.97      0.97      0.97        36

    accuracy                           0.98       100
   macro avg       0.98      0.98      0.98       100
weighted avg       0.98      0.98      0.98       100



# 4) 모델 저장하기

In [37]:
import joblib

joblib.dump(model, "rock_scissors_paper.pkl")


['rock_scissors_paper.pkl']

In [38]:
import joblib
model = joblib.load("rock_scissors_paper.pkl")

In [ ]:
import sys
import cv2
import mediapipe as mp
import os
import joblib
import numpy as np

#mediapipe hand landmark 옵션
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
# 모델 불러오기, 라벨 정의
model = joblib.load("rock_scissors_paper.pkl")
label = {"Rock": 0, "Scissors":1, "Paper":2}
labels = ["Rock", "Scissors", "Paper"]
# 웹캠
#    # 카메라 감지
vcap = cv2.VideoCapture(0)

while True:
    ret, frame = vcap.read()
    if not ret:
        print("카메라가 작동하지 않습니다.")
        sys.exit()
    # 좌우 반전
    frame = cv2.flip(frame, 1)
    # 손 그리기 준비
    frame.flags.writeable = True
    # 손 감지
    results = hands.process(frame)
    # Hand Landmark 추출
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            height, width, _ = frame.shape 
           # 손 하나의 Hand Landmark 추출
            data = []
            for landmark in hand_landmarks.landmark:
                #좌표 데이터
                data.extend([landmark.x, landmark.y, landmark.z])

                # 저장 데이터 만들기, 포인트 그리기
                point_x = int(landmark.x * width) # landmark.x 는 이미지에서의 비율적 위치(소수점)을 리턴하기 때문에 화면 넓이를 곱해줘야한다.
                point_y = int(landmark.y * height)

                cv2.circle(frame, (point_x, point_y), 5, (0, 255, 0), 2)
                # 예측
            pred = model.predict(np.array([data]))
            cv2.putText(frame, f"Your choice is {labels[pred[0]]}", (10, 50), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 255, 0), 2)

    # 화면 띄우기
    cv2.imshow("webcam", frame)
    # ECS 누르면 종료
    key = cv2.waitKey(1)
    if key == 27: #ESC
        break
    
# 마무리
vcap.release()
cv2.destroyAllWindows

c:\Users\user\potenup\python7month\CVproject\.venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


<function destroyAllWindows>

: 